In [2]:
# Checa GPU escolhida A100
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Mon Apr  6 02:28:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# Conecta-se com o Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Navega para a pasta onde estão os arquivos e programas
%cd /content/drive/My Drive/aval1_qabertas

/content/drive/My Drive/aval1_qabertas


In [12]:
# Instala pacote zstd necessário para baixar e executar o ollama no Colab
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (301 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently i

In [13]:
# Instala o binário do Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [14]:
# Instala bibliotecas auxiliares para rodar em background e interagir via Python
!pip install ollama aiohttp pandas

In [15]:
# 1. Para o servidor atual se estiver rodando, em caso de algum erro, se for necessário
#!pkill ollama

# 2. Define que o Ollama pode manter até 3 modelos na memória, pois comporta na GPU A100 da assinatura Colab Pro
# Caso a GPU escolhida não comporte os 3 modelos, usar OLLAMA_MAX_LOADED_MODELS=1 ou comentar, pois é o default
%env OLLAMA_MAX_LOADED_MODELS=3

# 3. Define por quanto tempo eles ficam na memória (ex: 60 minutos ou -1 para sempre)
# %env OLLAMA_KEEP_ALIVE=60m
%env OLLAMA_KEEP_ALIVE=-1

# 4. Inicia o servidor ollama em segundo plano e libera o Colab para a próxima tarefa
# O '&' no final joga o processo para o background e redireciona mensagens para /dev/null
!ollama serve > /dev/null 2>&1 &

env: OLLAMA_MAX_LOADED_MODELS=3
env: OLLAMA_KEEP_ALIVE=-1


In [16]:
# Checar se o ollama está rodando
!curl http://localhost:11434

Ollama is running

In [17]:
# Checar versão do ollama
!ollama --version

ollama version is 0.20.2


In [ ]:
# Baixa o modelo llama3.1:8b
!ollama pull llama3.1:8b

In [ ]:
# Baixa o modelo gemma3:12b
!ollama pull gemma3:12b

In [ ]:
# Configura a execução do modelo llama3.1:8b
%%writefile Modelfile_llama_safe
FROM llama3.1:8b
PARAMETER temperature 0
PARAMETER repeat_penalty 1.2
PARAMETER repeat_last_n 128
PARAMETER num_predict 512
# Forçar 100% na GPU
PARAMETER num_gpu 100
SYSTEM """Você é um assistente jurídico especializado. Responda de forma curta, técnica e direta. Não repita informações desnecessariamente."""

Overwriting Modelfile_llama_safe


In [ ]:
# Cria um alias do modelo llama3.1:8b para ser executado com base na configuração acima
# llama_safe:latest
!ollama create llama_safe -f Modelfile_llama_safe

In [ ]:
# Configura a execução do modelo gemma3:12b
%%writefile Modelfile_gemma_safe
FROM gemma3:12b
PARAMETER temperature 0
PARAMETER repeat_penalty 1.2
PARAMETER repeat_last_n 128
PARAMETER num_predict 512
# PARAMETER num_gpu 41
# Forçar 100% na GPU
PARAMETER num_gpu 100
SYSTEM """Você é um assistente jurídico especializado. Responda de forma curta, técnica e direta. Não repita informações desnecessariamente."""

Overwriting Modelfile_gemma_safe


In [ ]:
# Cria um alias do modelo gemma3:12b para ser executado com base na configuração acima
# gemma_safe:latest
!ollama create gemma_safe -f Modelfile_gemma_safe

In [ ]:
# Lista todos os modelos disponíveis, incluindo os novos modelos (latest) configurados para execução
!ollama list

NAME                 ID              SIZE      MODIFIED               
llama_safe:latest    fc98d67b9431    4.9 GB    Less than a second ago    
gemma3:12b           f4031aab637d    8.1 GB    Less than a second ago    
gemma_safe:latest    2b9cd819acd6    8.1 GB    Less than a second ago    
llama3.1:8b          46e0c10c039e    4.9 GB    1 second ago              
jurema:latest        ae43edb604d5    4.7 GB    5 hours ago               


In [ ]:
# O modelo Jurema não tem dispoível na biblioteca do Ollama
# O seu arquivo GGUF foi baixado do Hugging Face de disponibilizado no meu drive
# Fazendo uma cópia para do meu Drive para a pasta /content do ambiente Colab, a fim de ficar mais rápido
# do que ler diretamente da minha pasta do Drive durante a execução
!cp /content/drive/MyDrive/aval1_qabertas/jurema-7b-q4_k_m.gguf /content/jurema.gguf

In [ ]:
# Configura a execução do modelo jurema-7b-q4_k_m.gguf renomeado acima para jurema.gguf
%%writefile Modelfile_jurema
FROM /content/jurema.gguf
PARAMETER temperature 0
# 1. Penalidade de repetição (Essencial para temperatura 0)
PARAMETER repeat_penalty 1.2
# 2. Janela de observação de repetição
PARAMETER repeat_last_n 128
# 3. Limite de tokens para evitar respostas infinitas
PARAMETER num_predict 512
# 4. Garante que use a GPU para não levar 6 minutos
# PARAMETER num_gpu 33
# Forçar 100% na GPU
PARAMETER num_gpu 100
SYSTEM """Você é um assistente jurídico especializado. Responda de forma curta, técnica e direta. Não repita informações desnecessariamente."""

#SYSTEM """Você é um assistente jurídico especializado em Direito Brasileiro.
#Responda de forma direta e técnica. Não repita a mesma conclusão várias vezes."""

Overwriting Modelfile_jurema


In [ ]:
# Cria um alias do modelo jurema.gguf para ser executado com base na configuração acima
!ollama create jurema -f Modelfile_jurema

In [ ]:
# Lista novamente todos os modelos disponíveis, incluindo os novos modelos (latest) configurados para execução
# desta feita incluindo o juremma:latest
!ollama list

NAME                 ID              SIZE      MODIFIED               
jurema:latest        ae43edb604d5    4.7 GB    Less than a second ago    
llama_safe:latest    fc98d67b9431    4.9 GB    21 seconds ago            
gemma3:12b           f4031aab637d    8.1 GB    22 seconds ago            
gemma_safe:latest    2b9cd819acd6    8.1 GB    21 seconds ago            
llama3.1:8b          46e0c10c039e    4.9 GB    22 seconds ago            


In [ ]:
# Testando o modelo jurema:latest
!time ollama run jurema:latest "Explique o que é o princípio da dignidade da pessoa humana."

 O princípio da dignidade da pessoa humana, também conhecido como valor int
intrínseco do ser humano ou humanismo, é um dos fundamentos centrais das Co
Constituições modernas e está presente em diversas legislações ao redor do 
mundo.

Este princípio estabelece a ideia de que todos os indivíduos possuem uma di
dignidade inerente simplesmente por existirem. Isso significa que cada pess
pessoa tem direitos básicos que devem ser respeitados independentemente de 
qualquer condição social, econômica ou cultural. A dignidade da pessoa huma
humana implica na proteção contra abusos e violações aos direitos fundament
fundamentais, garantindo o tratamento justo e igualitário para todas as pes
pessoas.

A importância deste princípio reside no fato de que ele serve como base par
para outros valores constitucionais, como liberdade, igualdade e justiça. A
Ao reconhecer a dignidade intrínseca do indivíduo, busca-se promover um amb
ambiente onde todos possam viver com autonomia, segurança jurídica e r

In [ ]:
# Testando o modelo llama_safe:latest
!time ollama run llama_safe:latest "Explique o que é o princípio da dignidade da pessoa humana."

O Princípio da Dignidade da Pessoa Humana (PDPH) é uma norma constitucional
constitucional fundamental, presente em vários ordenamentos jurídicos, espe
especialmente no Brasil através do artigo 1º, inciso III da Constituição Fe
Federal. Este princípio estabelece a proteção à integridade física, moral e
e psicológica das pessoas, garantindo-lhes respeito como seres humanos.

Ele se manifesta de várias maneiras:

- Protege contra tratamento desumano ou degradante;
- Garantia dos direitos fundamentais inerentes ao ser humano;
- Proibição de tortura e maus-tratos;
- Respeito pela autonomia individual.
O PDPH é considerado um limite máximo para as intervenções do Estado na esf
esfera privada das pessoas, servindo como parâmetro para interpretação da C
Constituição.


real	0m5.421s
user	0m0.030s
sys	0m0.027s


In [ ]:
# Testando o modelo gemma_safe:latest
!time ollama run gemma_safe:latest "Explique o que é o princípio da dignidade da pessoa humana."

É o fundamento do ordenamento jurídico brasileiro (art. 1º, III, CF/88), qu
que reconhece a cada indivíduo, inerentemente, o direito a condições materi
materiais mínimas para uma vida livre, plena e com desenvolvimento integral
integral. É valor superior, limitador do poder estatal e guia para a interp
interpretação das normas.


real	0m9.045s
user	0m0.031s
sys	0m0.030s


In [ ]:
# Verificando onde os modelos estão sendo executados, se 100% na GPU ou parcialmente CPU/GPU ou completamente CPU
!ollama ps

NAME                 ID              SIZE      PROCESSOR    CONTEXT    UNTIL   
gemma_safe:latest    2b9cd819acd6    18 GB     100% GPU     131072     Forever    
llama_safe:latest    fc98d67b9431    30 GB     100% GPU     131072     Forever    
jurema:latest        ae43edb604d5    8.2 GB    100% GPU     32768      Forever    


In [19]:
# Instalar as dependências necessárias para o cálculo da métrica BERTScore e geração de graficos
!pip -q install bert-score transformers sentencepiece requests pandas tqdm matplotlib accelerate tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.7 MB/s eta 0:00:00


In [24]:
# Rodar programa 01
# Submete as 10 questões aos 3 modelos e calcula o BERTScore
!python3 program_01_openq.py

Failed checking if argv[0] is an import path entry
Traceback (most recent call last):
  File "<frozen zipimport>", line 92, in __init__
KeyError: '/content/drive/MyDrive/aval1_qabertas/program_01_openq.py'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "<frozen zipimport>", line 94, in __init__
  File "<frozen zipimport>", line 368, in _read_directory
KeyboardInterrupt
ERROR: Operation cancelled by user
^C


In [6]:
# Gerar os gráficos
!python3 program_02_openq.py

Figure(700x600)
Figure(700x600)
Figure(700x600)
Figure(700x600)
Figure(700x600)
Figure(700x600)
Figure(700x600)
Figure(700x600)
Figure(700x600)
Figure(700x600)
Figure(700x600)
Figure(700x600)
Figure(700x600)
Figure(700x600)
Figure(700x600)
